In [64]:
import math
import re
import pandas as pd

In [65]:
df = pd.read_csv('train_set_final.csv', delimiter='\t')

In [66]:
texts = []
for i in range(50000):
    texts.append(df.iloc[i].to_dict()['Title']+' '+df.iloc[i].to_dict()['Subtitle'])
len(texts)

50000

In [77]:
texts[0]

'Listening and promising has become overrated I have come across situations where Ive clearly stated how I feel and have also repeated it a bunch of times but I dont know why people these days have selective hearing. I know I tend to do that but never forget important things that are like'

In [68]:
tags = []
for i in range(50000):
    tags.append(df.iloc[i].to_dict()['Tags'])
len(tags)

50000

In [69]:
tags[:2]

['life-lessons', 'web-development']

In [70]:
type(texts[1])

str

In [97]:
def tokenize(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)  # удаляем знаки препинания
    tokens = text.split()
    return tokens

def generate_ngrams(tokens, n):
    ngrams = []
    for i in range(len(tokens) - n + 1):
        ngram = ' '.join(tokens[i:i+n])
        ngrams.append(ngram)
    return ngrams

In [102]:
tokens = tokenize(texts[0])
ngrams = generate_ngrams(tokens, 2)
tokens = tokens + ngrams

tokens[-1]

'are like'

In [110]:
class NaiveBayesClassifier:
    def __init__(self, use_ngrams=False, n=1):
        self.class_counts = {} # {"positive": 3, "negative": 3}
        self.word_counts = {} # {"positive": {"love": 2, "movie": 3, ...}
        self.class_total_words = {} # {"positive": 50, "negative": 45}
        self.vocabulary = set()
        self.total_docs = 0
        self.use_ngrams = use_ngrams
        self.n = n

    def fit(self, texts, tags):
        self.total_docs = len(texts)
        for text, tag in zip(texts, tags):
            # print(text, '\n', tag, '\n\n\n\n')
            if tag not in self.class_counts:
                self.class_counts[tag] = 0
                self.word_counts[tag] = {}
                self.class_total_words[tag] = 0
            self.class_counts[tag] += 1

            tokens = tokenize(text)
            if self.use_ngrams and self.n > 1:
                ngrams = generate_ngrams(tokens, self.n)
                tokens = ngrams
            
            for token in tokens:
                self.vocabulary.add(token)
                self.word_counts[tag][token] = self.word_counts[tag].get(token, 0) + 1
                self.class_total_words[tag] += 1

    def predict(self, texts):
        predictions = []
        for text in texts:
            tokens = tokenize(text)
            if self.use_ngrams and self.n > 1:
                ngrams = generate_ngrams(tokens, self.n)
                tokens = ngrams
            class_scores = {}
            for tag in self.class_counts:
                log_prob = math.log(self.class_counts[tag] / self.total_docs) # лог для сложения вместо умножения
                for token in tokens:
                    token_count = self.word_counts[tag].get(token, 0)
                    log_prob += math.log((token_count + 1) / (self.class_total_words[tag] + len(self.vocabulary)))
                class_scores[tag] = log_prob
            predicted_label = max(class_scores, key=class_scores.get)
            predictions.append(predicted_label)
        return predictions

In [111]:
texts[:2][1]

'Wix Website The Wix website builder is an online software that enables anyone with access to the internet to design a quick and basic website, based on pre-designed templates. The graphic editor interface is straightforward enough for anybody to customize a free website, with the aid of Wixs archive of templates'

In [112]:
model = NaiveBayesClassifier(use_ngrams=True, n=2)
model.fit(texts, tags)

In [114]:
test_text = [
    'Wix Website The Wix website builder is an online software that enables anyone with access to the internet to design a quick and basic website, based on pre-designed templates. The graphic editor interface is straightforward enough for anybody to customize a free website, with the aid of Wixs archive of templates'
]
predictions = model.predict(test_text)
for text, label in zip(test_text, predictions):
        print(f"Текст: {text}\nПредсказанная метка: {label}\n")

Текст: Wix Website The Wix website builder is an online software that enables anyone with access to the internet to design a quick and basic website, based on pre-designed templates. The graphic editor interface is straightforward enough for anybody to customize a free website, with the aid of Wixs archive of templates
Предсказанная метка: web-development

